# Segment correlation distance

Turns the Pearson correlation matrix behind the segment graph into the correlation distance

$$D_{ij} = \sqrt{2\,(1 - r_{ij})}$$

and checks it against the graph the service loads (`segment_correlation_graph.json`, written by
`backend/export_segment_correlation.py`).

- For series standardised to mean 0 and variance 1, $D_{ij}$ is proportional to the Euclidean distance between them,
  so it is a true metric (symmetric, zero on the diagonal, triangle inequality). The correlations themselves are not,
  which is why clustering or multidimensional scaling of the states should run on $D$ rather than on $r$.
- $D$ falls as $r$ rises, so the graph's rule $r \ge 0.95$ is the same rule as $D \le \sqrt{2(1-0.95)} \approx 0.316$.
  The edge-set check below is therefore a check that the series still reproduces the shipped graph, not a new finding.

Input: `data/docs/segment_monthly_series.parquet` (built by `backend/build_segment_series.py`; not in the repository).
Set `SEGMENT_SERIES` to read a copy elsewhere. Outputs go to `results/` next to this notebook.

In [1]:
import json
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

THRESHOLD = 0.95  # same as backend/export_segment_correlation.py

repo_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "backend" / "risk-engine").is_dir())
series_path = Path(os.environ.get("SEGMENT_SERIES", repo_root / "data" / "docs" / "segment_monthly_series.parquet"))
graph_path = repo_root / "backend" / "risk-engine" / "src" / "main" / "resources" / "segment_correlation_graph.json"
results = repo_root / "backend" / "experiments" / "segment_correlation_distance" / "results"
results.mkdir(exist_ok=True)

if not series_path.is_file():
    raise FileNotFoundError(f"{series_path} not found: build it with backend/build_segment_series.py or set SEGMENT_SERIES")
print("series:", series_path.name)

series: segment_monthly_series.parquet


## Load the state-by-month series

In [2]:
series = pd.read_parquet(series_path)
key = ["property_state", "monthly_reporting_period"]
assert {*key, "delinquency_rate"} <= set(series.columns), list(series.columns)
assert not series.duplicated(key).any(), "duplicate (state, month) rows"

# pivot, not pivot_table: a duplicate would be averaged silently by pivot_table
wide = series.pivot(index="monthly_reporting_period", columns="property_state", values="delinquency_rate").sort_index()
assert not wide.isna().any().any(), "a state is missing months"
states = list(wide.columns)
print(f"{len(states)} states x {len(wide)} months, {wide.index.min()} .. {wide.index.max()}")

15 states x 69 months, 2020-01-01 .. 2025-09-01


## Correlation and distance matrices

In [3]:
corr = wide.corr(method="pearson")
# clip guards against r a hair above 1 from rounding; the diagonal is set to exactly 0
d = np.sqrt(np.clip(2 * (1 - corr.to_numpy()), 0, None))
np.fill_diagonal(d, 0.0)
dist = pd.DataFrame(d, index=states, columns=states)

off = ~np.eye(len(states), dtype=bool)
# triangle inequality over every ordered triple: d[i,k] <= d[i,j] + d[j,k]
triangle_slack = (d[:, None, :] - d[:, :, None] - d.T[None, :, :]).max()
properties = {
    "symmetric": bool(np.allclose(corr, corr.T, atol=1e-12)),
    "corr_diagonal_one": bool(np.allclose(np.diag(corr), 1.0, atol=1e-12)),
    "distance_diagonal_zero": bool((np.diag(d) == 0).all()),
    "distance_nonnegative": bool((d >= 0).all()),
    "triangle_inequality": bool(triangle_slack <= 1e-12),
    "pearson_offdiagonal_min": float(corr.to_numpy()[off].min()),
    "pearson_offdiagonal_max": float(corr.to_numpy()[off].max()),
    "distance_offdiagonal_min": float(d[off].min()),
    "distance_offdiagonal_max": float(d[off].max()),
}
assert all(v for k, v in properties.items() if isinstance(v, bool)), properties
properties

{'symmetric': True,
 'corr_diagonal_one': True,
 'distance_diagonal_zero': True,
 'distance_nonnegative': True,
 'triangle_inequality': True,
 'pearson_offdiagonal_min': 0.36713326267020313,
 'pearson_offdiagonal_max': 0.9914915430046535,
 'distance_offdiagonal_min': 0.13044889417198235,
 'distance_offdiagonal_max': 1.125048209926843}

## Threshold equivalence and the shipped graph

In [4]:
cutoff = float(np.sqrt(2 * (1 - THRESHOLD)))
pairs = pd.DataFrame(
    [
        {"state_i": a, "state_j": b, "pearson_r": float(corr.loc[a, b]), "distance_d": float(dist.loc[a, b])}
        for i, a in enumerate(states)
        for b in states[i + 1:]
    ]
)
pairs["r_ge_threshold"] = pairs["pearson_r"] >= THRESHOLD
pairs["d_le_cutoff"] = pairs["distance_d"] <= cutoff
pairs = pairs.sort_values("distance_d", ignore_index=True)
assert pairs["r_ge_threshold"].equals(pairs["d_le_cutoff"])

graph = json.loads(graph_path.read_text(encoding="utf-8"))
shipped = {tuple(sorted((e["source"], e["target"]))): e["weight"] for e in graph["edges"]}
by_cutoff = {tuple(sorted((r.state_i, r.state_j))): round(r.pearson_r, 4) for r in pairs.itertuples() if r.d_le_cutoff}
graph_comparison = {
    "shipped_edges": len(shipped),
    "cutoff_edges": len(by_cutoff),
    "same_pairs": set(shipped) == set(by_cutoff),
    "same_weights_4dp": shipped == by_cutoff,
    "same_nodes": sorted(graph["nodes"]) == sorted(states),
}
assert all(v for v in graph_comparison.values() if isinstance(v, bool)), graph_comparison
print(f"cutoff D <= {cutoff:.6f}")
graph_comparison

cutoff D <= 0.316228


{'shipped_edges': 41,
 'cutoff_edges': 41,
 'same_pairs': True,
 'same_weights_4dp': True,
 'same_nodes': True}

## Write the tables and the report

In [5]:
corr.to_csv(results / "pearson_correlation_matrix.csv", float_format="%.10f")
dist.to_csv(results / "distance_matrix.csv", float_format="%.10f")
pairs.to_csv(results / "pairwise_correlation_distance.csv", index=False, float_format="%.10f")

report = {
    "input": {
        "series": "data/docs/segment_monthly_series.parquet",
        "graph": "backend/risk-engine/src/main/resources/segment_correlation_graph.json",
        "rows": len(series),
        "states": states,
        "months": len(wide),
        "month_start": str(wide.index.min()),
        "month_end": str(wide.index.max()),
    },
    "properties": properties,
    "threshold": {"pearson_r": THRESHOLD, "distance_cutoff": cutoff, "rules_agree": True},
    "graph_comparison": graph_comparison,
    "closest_pairs": pairs.head(5)[["state_i", "state_j", "pearson_r", "distance_d"]].to_dict("records"),
}
(results / "validation_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print("wrote", sorted(p.name for p in results.iterdir()))

wrote ['correlation_vs_distance.png', 'distance_distribution.png', 'distance_matrix.csv', 'distance_matrix_heatmap.png', 'pairwise_correlation_distance.csv', 'pearson_correlation_heatmap.png', 'pearson_correlation_matrix.csv', 'threshold_equivalence.png', 'validation_report.json']


## Charts

In [6]:
def save(fig, name):
    fig.tight_layout()
    fig.savefig(results / name, dpi=120)
    plt.close(fig)


for frame, name, title, cmap in [
    (corr, "pearson_correlation_heatmap.png", "Pearson r of monthly delinquency rates", "viridis"),
    (dist, "distance_matrix_heatmap.png", "Correlation distance D = sqrt(2(1 - r))", "viridis_r"),
]:
    fig, ax = plt.subplots(figsize=(9, 7.5))
    sns.heatmap(frame, annot=True, fmt=".2f", annot_kws={"size": 7}, cmap=cmap, square=True, ax=ax)
    ax.set_title(title)
    save(fig, name)

r_grid = np.linspace(min(0.3, pairs["pearson_r"].min()), 1, 200)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(r_grid, np.sqrt(2 * (1 - r_grid)), color="0.6", label="D = sqrt(2(1 - r))")
ax.scatter(pairs["pearson_r"], pairs["distance_d"], c=pairs["d_le_cutoff"].map({True: "C3", False: "C0"}), s=18)
ax.axvline(THRESHOLD, ls="--", color="0.3")
ax.axhline(cutoff, ls=":", color="0.3")
ax.set_xlabel("Pearson r")
ax.set_ylabel("distance D")
ax.set_title(f"State pairs: {int(pairs['d_le_cutoff'].sum())} of {len(pairs)} within the graph cutoff (red)")
ax.legend()
save(fig, "correlation_vs_distance.png")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(pairs["distance_d"], bins=25, color="C0")
ax.axvline(cutoff, ls="--", color="C3", label=f"cutoff {cutoff:.3f}")
ax.set_xlabel("distance D")
ax.set_ylabel("state pairs")
ax.set_title("Distribution of pairwise distances")
ax.legend()
save(fig, "distance_distribution.png")

edge_mask = pd.DataFrame(False, index=states, columns=states)
for a, b in shipped:
    edge_mask.loc[a, b] = edge_mask.loc[b, a] = True
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(edge_mask.astype(int) + (dist <= cutoff).astype(int) - np.eye(len(states), dtype=int),
            cmap=sns.color_palette(["#f0f0f0", "#e6a23c", "#2a7ab9"]), vmin=0, vmax=2, cbar=False, square=True, ax=ax)
ax.set_title("Shipped edge and D <= cutoff: blue = both, orange = one only")
save(fig, "threshold_equivalence.png")
print("charts written")

charts written
